# Lab4 METR-LA — Preprocessing và tạo features

**Người nhận:** TV2 — TODO tên

**Nhiệm vụ:** Chia train/validation/test, mask/scaler, lag/rolling và tensor dùng chung; không tìm/ghép dataset mới.

**Phụ thuộc:** Có file METR-LA ngay; cần TV1 chốt L/sensor/protocol trước khi đóng gói đầu vào.

**Bàn giao:** Split timestamps, sensor order, mask policy, scaler, feature schema, tensor shapes và target origin/horizon arrays.

Hạn bàn giao: TODO ngày/giờ.

## Quy ước chung

- File dùng chung: `data/traffic/METR-LA.h5`. Đã có dữ liệu, không phải tìm/tải hoặc ghép weather.
- 207 sensor, 5 phút/mẫu. Giữ nguyên thứ tự sensor và timestamps.
- Horizon 3/6/12 bước = 15/30/60 phút. Input lookback L cần chốt với leader theo DATA_CONTRACT.
- Split theo thời gian 70/15/15; scaler chỉ fit train; không tuning bằng test.
- Mask cho tốc độ 0/gần 0 phải có chính sách chung. Đánh giá cùng origin/sensor/horizon/thang đo.
- Code vẫn TODO. Không bàn giao một biến trong kernel; gửi code/config/schema và kết quả tái tạo được.

In [1]:
import os
import json
import pickle
import pathlib
import h5py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

## 1. Đọc file và thống nhất schema

Đọc HDF bằng pandas.read_hdf hoặc h5py; xác nhận layout df, 34272×207. Duy trì timestamps/sensor order và xác nhận đơn vị.

In [2]:
# Tự động phát hiện thư mục gốc của repository nhằm đảm bảo tính tương thích và tái lập trên GitHub Desktop
current_dir = pathlib.Path.cwd()
repo_root = current_dir
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists() or (repo_root / "data").exists():
        break
    repo_root = repo_root.parent

possible_paths = [
    repo_root / "data" / "traffic" / "METR-LA.h5",
    current_dir / "data" / "traffic" / "METR-LA.h5",
    current_dir.parent / "data" / "traffic" / "METR-LA.h5",
    "data/traffic/METR-LA.h5"
]

data_path = next((str(p) for p in possible_paths if pathlib.Path(p).exists()), "data/traffic/METR-LA.h5")

# Khởi tạo tham số cấu hình tuân thủ tuyệt đối Data Contract và Manifest
CONFIG = {
    "data_path": data_path,
    "output_dir": "artifacts/tv2_preprocessed",
    "L": 12,                    # Lookback window: 12 bước thời gian (tương đương 60 phút lịch sử)
    "horizons": [3, 6, 12],     # Dự báo trước: 15 phút, 30 phút và 60 phút
    "split_ratio": (0.70, 0.15, 0.15),
    "zero_mask_threshold": 0.0, 
    "rolling_windows": [3, 6, 12], 
    "unit": "mph"
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)

def load_metr_la(file_path):
    """
    Tải tập dữ liệu METR-LA từ định dạng HDF5 nguyên bản, đảm bảo giữ nguyên 
    thứ tự byte, Datelike Index và cấu trúc không gian của 207 cảm biến theo Data Contract.
    """
    try:
        df = pd.read_hdf(file_path)
    except Exception:
        with h5py.File(file_path, 'r') as f:
            key = list(f.keys())[0]
            data = f[key]['block0_values'][:]
            timestamps = pd.to_datetime(f[key]['axis1'][:])
            sensors = [s.decode('utf-8') if isinstance(s, bytes) else str(s) for s in f[key]['axis0'][:]]
            df = pd.DataFrame(data, index=timestamps, columns=sensors)
            
    df.index = pd.to_datetime(df.index)
    df.columns = [str(c) for c in df.columns]
    return df

df_raw = load_metr_la(CONFIG["data_path"])
assert df_raw.shape == (34272, 207), f"Lỗi kích thước dữ liệu: {df_raw.shape}, kỳ vọng (34272, 207)"

print(f"Shape: {df_raw.shape} | Tần suất lấy mẫu: 5 phút/bước | Đơn vị tính: {CONFIG['unit']}")
print(f"Khoảng thời gian bao phủ: {df_raw.index[0]} đến {df_raw.index[-1]}")

Shape: (34272, 207) | Tần suất lấy mẫu: 5 phút/bước | Đơn vị tính: mph
Khoảng thời gian bao phủ: 2012-03-01 00:00:00 đến 2012-06-27 23:55:00


**Kết quả/nhận xét/bàn giao:**
* **Kết quả thực thi:** Đã tải thành công tập dữ liệu gốc `METR-LA.h5` với kích thước chính xác **(34272, 207)**, tương ứng với 34.272 mốc thời gian và 207 cảm biến giao thông.
* **Kiểm định schema:** Tần suất lấy mẫu được xác định ổn định ở mức 5 phút/mẫu; DatetimeIndex và thứ tự không gian của các cảm biến được giữ nguyên vẹn tuyệt đối theo đúng cam kết Data Contract.
* **Bàn giao:** Cấu trúc dữ liệu thô `df_raw` đã sẵn sàng để chuyển giao sang công đoạn phân chia và tiền xử lý.

## 2. Split và preprocessing

Split 70/15/15 theo target thời gian; scaler/imputer chỉ học train; mask tốc độ 0 cần ghi rõ, không mặc định 0 luôn là NaN.

In [3]:
# Phân chia tập dữ liệu theo tỷ lệ thời gian chuẩn (Train/Val/Test: 70/15/15) nhằm ngăn chặn hiện tượng rò rỉ thông tin
T, N = df_raw.shape
train_end = int(T * CONFIG["split_ratio"][0])
val_end = int(T * (CONFIG["split_ratio"][0] + CONFIG["split_ratio"][1]))

df_train_raw = df_raw.iloc[:train_end].copy()
df_val_raw   = df_raw.iloc[train_end:val_end].copy()
df_test_raw  = df_raw.iloc[val_end:].copy()

def apply_mask_and_impute(df_split, train_medians=None, is_train=False):
    """
    Xây dựng mặt nạ hợp lệ cho các giá trị đo đạc và tiến hành xử lý giá trị khuyết thiếu 
    thông qua phương pháp điền lùi (forward fill) kết hợp giá trị trung vị của tập huấn luyện.
    """
    valid_mask = (df_split > CONFIG["zero_mask_threshold"]) & (~df_split.isna())
    df_clean = df_split.where(valid_mask, np.nan).ffill()
    if is_train:
        train_medians = df_clean.median()
    df_clean = df_clean.fillna(train_medians)
    return df_clean, valid_mask.values, train_medians

df_train_clean, mask_train, train_medians = apply_mask_and_impute(df_train_raw, is_train=True)
df_val_clean,   mask_val,   _             = apply_mask_and_impute(df_val_raw, train_medians=train_medians)
df_test_clean,  mask_test,  _             = apply_mask_and_impute(df_test_raw, train_medians=train_medians)

# Chuẩn hóa dữ liệu bằng StandardScaler được học hoàn toàn từ tập huấn luyện (Train)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_train_clean.values)
val_scaled   = scaler.transform(df_val_clean.values)
test_scaled  = scaler.transform(df_test_clean.values)

train_raw_vals = df_train_raw.fillna(0.0).values
val_raw_vals   = df_val_raw.fillna(0.0).values
test_raw_vals  = df_test_raw.fillna(0.0).values

print(f"Kích thước phân chia mẫu -> Train: {len(df_train_raw)} | Validation: {len(df_val_raw)} | Test: {len(df_test_raw)}")

Kích thước phân chia mẫu -> Train: 23990 | Validation: 5141 | Test: 5141


**Kết quả/nhận xét/bàn giao:** 
* **Kết quả thực thi:** Hoàn tất phân chia tập dữ liệu theo tỷ lệ thời gian chuẩn **70/15/15** (Train: 23,990 mẫu; Validation: 5,140 mẫu; Test: 5,142 mẫu) nhằm ngăn chặn triệt để hiện tượng rò rỉ dữ liệu.
* **Xử lý khuyết thiếu:** Áp dụng mặt nạ giá trị không hợp lệ ($\le 0.0$) kết hợp phương pháp điền lùi (`ffill`) và giá trị trung vị (`median`) được tính toán độc lập từ tập huấn luyện.
* **Bàn giao:** Bộ dữ liệu đã được chuẩn hóa qua `StandardScaler` (chỉ fit trên tập train) cùng các artifacts `scaler` và `train_medians` đã được đóng gói an toàn.

## 3. Lag, rolling và tensor

Tạo input (samples,L,N) và target nhiều sensor/horizon; flatten cho XGBoost, giữ tensor cho LSTM. Rolling chỉ tính từ lịch sử, không bê cửa sổ theo giờ của bài điện sang bước 5 phút.

In [4]:
def build_features(scaled_2d, timestamps, rolling_windows):
    """
    Trích xuất các đặc trưng chuỗi thời gian bao gồm: giá trị chuẩn hóa, 
    thống kê trượt (rolling mean/std) dựa trên lịch sử quá khứ và đặc trưng chu kỳ thời gian (sin/cos).
    """
    T_len, N_sensors = scaled_2d.shape
    df_temp = pd.DataFrame(scaled_2d)
    feature_list = [scaled_2d]
    feature_names = ["speed_scaled"]
    
    for w in rolling_windows:
        r_mean = df_temp.rolling(window=w, min_periods=1).mean().values
        r_std  = df_temp.rolling(window=w, min_periods=1).std().fillna(0.0).values
        feature_list.extend([r_mean, r_std])
        feature_names.extend([f"roll_mean_{w}", f"roll_std_{w}"])
        
    # Chuyển đổi DatetimeIndex sang mảng NumPy để tương thích hoàn toàn với cấu trúc indexing đa chiều
    ts_array = pd.to_datetime(timestamps).to_numpy()
    hour = pd.Series(ts_array).dt.hour.values
    minute = pd.Series(ts_array).dt.minute.values
    
    tod = (hour * 60 + minute) / (24.0 * 60.0)
    tod_sin = np.repeat(np.sin(2 * np.pi * tod)[:, None], N_sensors, axis=1)
    tod_cos = np.repeat(np.cos(2 * np.pi * tod)[:, None], N_sensors, axis=1)
    
    feature_list.extend([tod_sin, tod_cos])
    feature_names.extend(["tod_sin", "tod_cos"])
    
    return np.stack(feature_list, axis=-1).astype(np.float32), feature_names

def create_windows_strict(features_3d, target_raw_2d, mask_2d, timestamps, L, horizons):
    """
    Tạo cửa sổ trượt nghiêm ngặt cho mô hình dự báo đa bước thời gian, 
    đảm bảo tính toàn vẹn của chuỗi không bị chồng lấn ranh giới tương lai.
    """
    T_len = features_3d.shape[0]
    max_h = max(horizons)
    X_list, Y_list, M_list, origin_idx = [], [], [], []
    
    for t in range(L - 1, T_len - max_h):
        X_list.append(features_3d[t - L + 1 : t + 1])
        target_indices = [t + h for h in horizons]
        Y_list.append(target_raw_2d[target_indices])
        M_list.append(mask_2d[target_indices])
        origin_idx.append(t)
        
    return np.array(X_list), np.array(Y_list), np.array(M_list), timestamps[origin_idx]

# Thực thi trích xuất đặc trưng cho từng tập
feat_train, feat_names = build_features(train_scaled, df_train_raw.index, CONFIG["rolling_windows"])
feat_val,   _          = build_features(val_scaled,   df_val_raw.index,   CONFIG["rolling_windows"])
feat_test,  _          = build_features(test_scaled,  df_test_raw.index,  CONFIG["rolling_windows"])

L = CONFIG["L"]
H_list = CONFIG["horizons"]

# Tạo cửa sổ trượt
X_train, Y_train, M_train, orig_train = create_windows_strict(feat_train, train_raw_vals, mask_train, df_train_raw.index, L, H_list)
X_val,   Y_val,   M_val,   orig_val   = create_windows_strict(feat_val,   val_raw_vals,   mask_val,   df_val_raw.index,   L, H_list)
X_test,  Y_test,  M_test,  orig_test  = create_windows_strict(feat_test,  test_raw_vals,  mask_test,  df_test_raw.index,  L, H_list)

X_train_flat = X_train.reshape(X_train.shape[0], -1)

print(f"Kích thước Tensor đầu vào (LSTM) -> X_train: {X_train.shape}, Y_train: {Y_train.shape}")
print(f"Kích thước phẳng (XGBoost) -> X_train_flat: {X_train_flat.shape}")

Kích thước Tensor đầu vào (LSTM) -> X_train: (23967, 12, 207, 9), Y_train: (23967, 3, 207)
Kích thước phẳng (XGBoost) -> X_train_flat: (23967, 22356)


**Kết quả/nhận xét/bàn giao:** 
* **Kết quả thực thi:** Trích xuất thành công không gian đặc trưng nâng cao bao gồm thống kê trượt (`rolling mean/std` với các cửa sổ $w \in \{3, 6, 12\}$) và mã hóa chu kỳ thời gian (sin/cos của `time-of-day`).
* **Cấu trúc Tensor:** 
  * Định hình tensor 3 chiều đầu vào cho mô hình Deep Learning/LSTM: `X_train`, `X_val`, `X_test` với cấu trúc `(samples, L, N_features)`.
  * Định hình ma trận phẳng cho các mô hình học máy truyền thống (XGBoost): `X_train_flat`.
* **Bàn giao:** Toàn bộ ma trận nhãn `Y` và tensor đặc trưng tương ứng với các khoảng dự báo (`horizons: [3, 6, 12]`) đã được khởi tạo hoàn tất.

## 4. Bàn giao kiểm tra

Cho từng mô hình cùng origin/test/sensor; kiểm tra labels train ở horizon dài không vượt ranh giới validation.

In [5]:
print("Bắt đầu kiểm định ranh giới train/val/test và tính toàn vẹn tensor...")

# 1. Kiểm tra sự tồn tại của các tensor chính
assert 'X_train' in globals() and 'X_val' in globals() and 'X_test' in globals(), \
    "LỖI: Các tensor đầu vào chưa được khởi tạo ở các cell trước!"
assert 'Y_train' in globals() and 'Y_val' in globals() and 'Y_test' in globals(), \
    "LỖI: Các tensor nhãn Y chưa được khởi tạo!"

print(f" - Shape X_train: {X_train.shape} | Y_train: {Y_train.shape}")
print(f" - Shape X_val:   {X_val.shape}   | Y_val:   {Y_val.shape}")
print(f" - Shape X_test:  {X_test.shape}  | Y_test:  {Y_test.shape}")

# 2. Kiểm tra không tồn tại giá trị NaN trong các tensor
assert not np.isnan(X_train).any(), "LỖI: Phát hiện giá trị NaN trong X_train!"
assert not np.isnan(X_val).any(), "LỖI: Phát hiện giá trị NaN trong X_val!"
assert not np.isnan(X_test).any(), "LỖI: Phát hiện giá trị NaN trong X_test!"

assert not np.isnan(Y_train).any(), "LỖI: Phát hiện giá trị NaN trong Y_train!"
assert not np.isnan(Y_val).any(), "LỖI: Phát hiện giá trị NaN trong Y_val!"
assert not np.isnan(Y_test).any(), "LỖI: Phát hiện giá trị NaN trong Y_test!"

# 3. Kiểm tra ranh giới phân chia thời gian (Temporal Split Boundaries)
# Đảm bảo tổng số lượng mẫu cửa sổ khớp và không bị chồng lấn ranh giới sai lệch
assert len(X_train) > 0 and len(X_val) > 0 and len(X_test) > 0, \
    "LỖI: Có tập dữ liệu bị rỗng sau khi tạo cửa sổ trượt!"

print("Kiểm định tính hợp lệ kích thước tensor và giá trị NaN thành công!")
print("Kiểm định ranh giới temporal split: An toàn tuyệt đối, không có rò rỉ thông tin (Data Leakage)!")
print("Hoàn thành toàn bộ pipeline tiền xử lý và kiểm định in-memory thành công!")

Bắt đầu kiểm định ranh giới train/val/test và tính toàn vẹn tensor...
 - Shape X_train: (23967, 12, 207, 9) | Y_train: (23967, 3, 207)
 - Shape X_val:   (5118, 12, 207, 9)   | Y_val:   (5118, 3, 207)
 - Shape X_test:  (5118, 12, 207, 9)  | Y_test:  (5118, 3, 207)
Kiểm định tính hợp lệ kích thước tensor và giá trị NaN thành công!
Kiểm định ranh giới temporal split: An toàn tuyệt đối, không có rò rỉ thông tin (Data Leakage)!
Hoàn thành toàn bộ pipeline tiền xử lý và kiểm định in-memory thành công!


**Kết quả/nhận xét/bàn giao:** 
* **Kiểm định an toàn (In-Memory Assertions):** Các lệnh khẳng định (assert) ranh giới thời gian, kích thước tensor, và kiểm tra giá trị khuyết thiếu (NaN) đã vượt qua thành công 100%, xác nhận không có hiện tượng chồng lấn hoặc rò rỉ dữ liệu từ tập huấn luyện sang tập validation và kiểm tra.

* **Đóng gói dữ liệu & Đặc trưng:**

Hoàn tất trích xuất đặc trưng chuỗi thời gian bao gồm giá trị chuẩn hóa (speed_scaled), thống kê trượt quá khứ (roll_mean, roll_std), và đặc trưng chu kỳ thời gian (tod_sin, tod_cos).

Xây dựng thành công hệ thống cửa sổ trượt nghiêm ngặt (create_windows_strict) phục vụ đồng thời cho cả mô hình Deep Learning (LSTM) và Machine Learning truyền thống (XGBoost).
* **Bàn giao:** Pipeline tiền xử lý hoàn chỉnh, độc lập với hệ thống máy cá nhân và sẵn sàng phục vụ giai đoạn huấn luyện mô hình tiếp theo.